![](/Workspace/Users/sunnygupta2508@gmail.com/Databricks-Certified-Data-Engineer-Pro/Includes/images/deletes.png)

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
from pyspark.sql import functions as F

schema = "customer_id STRING, email STRING, first_name STRING, last_name STRING, gender STRING, street STRING, city STRING, country_code STRING, row_status STRING, row_time timestamp"

(
    spark.readStream
            .table("bronze")
            .filter("topic = 'customers'")
            .select(F.from_json(F.col("value").cast("string"), schema).alias("data"))
            .select("data.*", F.col('data.row_time').alias("request_timestamp"))
            .filter("row_status = 'delete'")
            .select("customer_id", "request_timestamp", 
                    F.date_add("request_timestamp",30).alias("deadline"),
                    F.lit("requested").alias("status"))
        .writeStream
            .outputMode("append")
            .option("checkpointLocation", f"{bookstore.checkpoint_path}/delete_requests")
            .trigger(availableNow=True)
            .table("delete_requests")
)

In [0]:
%sql
select * from delete_requests

In [0]:
%sql
delete from customers_silver where customer_id in (select customer_id from delete_requests where status = 'requested')

In [0]:
delete_df = (
    spark.readStream
            .format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion",2)
            .table("customers_silver")
)

In [0]:
def process_deletes(microBatchDF, batchId):
    microBatchDF.filter("_change_type = 'delete'").createOrReplaceTempView("deletes")

    microBatchDF.sparkSession.sql("""
                                  delete from customers_orders where customer_id in (select customer_id from deletes)""")
    
    microBatchDF.sparkSession.sql("""
                                  merge into delete_requests a
                                  using deletes b
                                  on a.customer_id = b.customer_id
                                  when matched then update set a.status = 'deleted'
                                  """)

In [0]:
(delete_df.writeStream
            .foreachBatch(process_deletes)
            .option("checkpointLocation",f"{bookstore.checkpoint_path}/deletes")
            .trigger(availableNow=True)
            .start()    
)


In [0]:
%sql
SELECT * FROM delete_requests

In [0]:
%sql
DESCRIBE HISTORY customers_orders